## Notebook para runtime de testes do modelo

Este notebook serve apenas como "servidor" temporário para rodar o modelo em desenvolvimento. O limite de GPU do colab é de até 12h/ notebook / conta, então deve ser gerado um novo notebook ao término do limite.

### Carregamento do modelo

In [1]:
# instalação de dependências para runtime
!pip install -U transformers accelerate huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 130.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 140.5 MB/s eta 0:00:00


In [ ]:
# logando no hugging face

from huggingface_hub import login
login(token=f"{HF_TOKEN}")  # logar com access token do hugging face para a org


In [3]:
# carregando modelo
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "fiap-hospital-helper/hospital-helper-qwen2.5-1.5b"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/2.51k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
# teste rápido para validação do carregamento

prompt = (
    "### Instrucao:\nResponda em pt-BR usando o contexto clinico fornecido.\n\n"
    "### Entrada:\nPergunta: Qual o protocolo para dor toracica?\n\n### Resposta:\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200, do_sample=False)
resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(resposta)

### Instrucao:
Responda em pt-BR usando o contexto clinico fornecido.

### Entrada:
Pergunta: Qual o protocolo para dor toracica?

### Resposta:
O Protocolo Clínico de Dor Torácica, desenvolvido pela Sociedade Brasileira de Pneumologia e Tussa (SBPT), é uma ferramenta importante que orienta os profissionais médicos sobre manejo clínico da dor torácica. Ele abrange diversas etapas do tratamento, desde a avaliação inicial até a recuperação completa. O documento destaca as principais causas de dor torácica, como pneumonia, tuberculose pulmonar, câncer pulmonar, entre outras condições, bem como sugere estratégias terapêuticas adequadas para cada caso específico. Além disso, ele oferece recomendações sobre monitorização e acompanhamento dos pacientes com dor torácica, incluindo medidas preventivas e cuidados paliativos. Este protocolo é essencial para garantir um atendimento eficaz e humanizado aos pacientes com dor torácica, contrib


### Montagem da API

In [6]:
# instalação de dependências

!pip install fastapi uvicorn pyngrok nest-asyncio -q

In [10]:
# importando token da conta ngrok (criar conta no ngrok para diponibilizar url - muda a cada reinicialização do notebook)
from google.colab import userdata # guardar o token na seção secrets do colab para não ficar hardcoded
ngrok_token = userdata.get('NGROK_TOKEN')

In [8]:
# montando a api
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio, uvicorn
from pyngrok import ngrok

app = FastAPI()

class GenerateRequest(BaseModel):
    pergunta: str
    contexto: str = ""
    max_new_tokens: int = 300

@app.post("/generate")
def generate(req: GenerateRequest):
    prompt = (
        "### Instrucao:\nResponda em pt-BR usando o contexto clinico fornecido.\n\n"
        f"### Entrada:\nPergunta: {req.pergunta}\n"
    )
    if req.contexto:
        prompt += f"Contexto:\n{req.contexto}\n"
    prompt += "\n### Resposta:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=req.max_new_tokens, do_sample=False)
    texto = tokenizer.decode(outputs[0], skip_special_tokens=True)
    resposta = texto.split("### Resposta:")[-1].strip()
    return {"resposta": resposta}

@app.get("/")
def health():
    return {"status": "ok"}

In [12]:
# criando tunel http para acesso à api
ngrok.set_auth_token(ngrok_token)  # token do ngrok (pegar a url gerada a cada reinicialização do server)

public_url = ngrok.connect(8000)
print("API pública disponível em:", public_url)

nest_asyncio.apply()

config = uvicorn.Config(app, port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

API pública disponível em: NgrokTunnel: "https://shining-setting-trench.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [1580]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     179.187.133.254:0 - "GET / HTTP/1.1" 200 OK
INFO:     179.187.133.254:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.182.254.100:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     34.182.254.100:0 - "GET / HTTP/1.1" 200 OK
INFO:     34.182.254.100:0 - "POST /generate HTTP/1.1" 422 Unprocessable Entity
INFO:     2804:18:d2:7489:1:0:157f:4cc6:0 - "GET / HTTP/1.1" 200 OK
INFO:     2804:18:d2:7489:1:0:157f:4cc6:0 - "GET /generate HTTP/1.1" 405 Method Not Allowed


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [1580]
